# Steam Reviews Data Acquisition and Preprocessing Pipeline

## Overview
This notebook implements an end-to-end data acquisition and preprocessing pipeline using Steam user reviews as the source of customer feedback data. The system retrieves review data from the Steam reviews endpoint, preprocesses the raw text, engineers useful features, and loads the processed output into a SQLite database.

## Aim
The aim of this project is to demonstrate a complete data pipeline that includes:
- data acquisition from an external source
- preprocessing and cleaning of text data
- feature extraction and transformation
- loading the processed data into a database
- testing through unit tests and integration tests

This project has been designed to satisfy the task requirement of building a data acquisition and preprocessing pipeline with appropriate transformations and database loading.

In [1]:
!pip install requests

## Step 1: Importing Required Libraries

This section imports the Python libraries required for the pipeline.

- `re` is used for text cleaning through regular expressions.
- `time` is used to pause requests during paginated retrieval.
- `sqlite3` is used to store processed data in a relational database.
- `requests` is used to retrieve review data from the Steam endpoint.
- `pandas` is used for tabular processing, transformation, and exporting data.

These libraries form the base of the acquisition, preprocessing, and storage workflow.

In [2]:
import re
import time
import sqlite3
import requests
import pandas as pd
from urllib.parse import quote

## Step 2: Data Acquisition from Steam

This function retrieves user review data from the Steam reviews endpoint using an application ID. Steam provides review data in JSON format, which makes it suitable for automated acquisition and preprocessing.

The function supports:
- selecting a specific Steam application by `app_id`
- limiting the maximum number of reviews retrieved
- filtering by language
- filtering by review type

The review data is collected using cursor-based pagination. This means the function can retrieve multiple batches of reviews until the requested number of reviews is reached or no more reviews are available.

The output of this function is a list of dictionaries, where each dictionary contains one review and its associated metadata.

In [3]:
#Extraction logic for this function was self-devised and given to ChatGPT for the syntax, for better speed, clarity and efficiency.
def get_steam_reviews(app_id, max_reviews=100, language="english", review_type="all"):


    """
    Fetch reviews from Steam using cursor-based pagination.
    Steam returns up to 100 reviews per page.
    """

    reviews = []
    cursor = "*"

    while len(reviews) < max_reviews:
        remaining = max_reviews - len(reviews)
        num_per_page = min(100, remaining)

        url = f"https://store.steampowered.com/appreviews/{app_id}"

        #setting parameters for API call
        parameters = {
            "json": 1,
            "language": language,
            "filter": "recent",
            "review_type": review_type,
            "purchase_type": "all",
            "num_per_page": num_per_page,
            "cursor": cursor
        }

        response = requests.get(url, params=parameters)
        response.raise_for_status()
        data = response.json()

        #check to ensure the API request went through successfully, if not, then break.
        if data.get("success") != 1:
            print("API connection unsuccessful.")
            break

        batch = data.get("reviews", [])
        if not batch:
            break

        for review in batch:
            reviews.append({
                "recommendationid": review.get("recommendationid"),
                "review_text": review.get("review"),
                "voted_up": review.get("voted_up"),
                "votes_up": review.get("votes_up"),
                "votes_funny": review.get("votes_funny"),
                "comment_count": review.get("comment_count"),
                "steam_purchase": review.get("steam_purchase"),
                "received_for_free": review.get("received_for_free"),
                "written_during_early_access": review.get("written_during_early_access"),
                "language": review.get("language"),
                "timestamp_created": review.get("timestamp_created"),
                "timestamp_updated": review.get("timestamp_updated"),
                "author_steamid": review.get("author", {}).get("steamid"),
                "author_num_games_owned": review.get("author", {}).get("num_games_owned"),
                "author_num_reviews": review.get("author", {}).get("num_reviews"),
                "playtime_forever": review.get("author", {}).get("playtime_forever"),
                "playtime_last_two_weeks": review.get("author", {}).get("playtime_last_two_weeks"),
                "playtime_at_review": review.get("author", {}).get("playtime_at_review")
            })

        cursor = data.get("cursor")
        if not cursor: #if no reviews are fetched, it breaks the process here.
            break

    return reviews


## Step 3: Text Cleaning Function

This function cleans the raw review text so that it becomes more suitable for analysis and transformation.

The cleaning process performs the following operations:
- removes HTML tags if present
- removes punctuation and special characters
- converts text to lowercase
- removes extra spaces

This step is necessary because raw user-generated reviews often contain inconsistent formatting, symbols, and noise that can reduce the quality of later analysis.

In [4]:
def clean_text(text):
    if not text: #check to ensure that the text is not null
        return ""

    text = re.sub(r"<.*?>", "", text) #removing all HTML tags
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text) #removing all punctuations
    text = re.sub(r"\s+", " ", text).strip() #removing all extra white spaces
    text = text.lower() #converting text to lower case

    return text

## Step 4: Handling Missing Values

This function removes rows where the main review text is missing.

Missing values can reduce the quality of the processed dataset and create problems during feature extraction. Since review text is the central field for this project, rows without review text are excluded from further processing.

## Step 5: Removing Duplicate Reviews

This function removes duplicate records from the dataset using the unique `recommendationid` provided by Steam.

Duplicate records can affect analysis and distort the stored dataset. Removing duplicates ensures that each review is only processed and stored once.

## Step 6: Feature Engineering and Transformation

This function creates new variables from the raw review data in order to make the dataset more useful for analysis.

The following features are generated:
- `cleaned_review`: the cleaned version of the original review text
- `review_length`: the number of words in the cleaned review
- `sentiment_label`: a simple sentiment class based on whether the review is recommended or not
- `review_date`: a readable datetime value created from the Unix timestamp

This step adds analytical value to the original data and demonstrates the transformation stage of the pipeline.

## Step 7: Preprocessing Pipeline

This function combines the main preprocessing stages into one pipeline.

The steps performed are:
1. convert the raw list of reviews into a pandas DataFrame
2. remove rows with missing review text
3. remove duplicate reviews
4. engineer additional features from the raw and cleaned data

This function is useful because it allows the preprocessing workflow to be reused in a single call after the data acquisition stage.

In [5]:
#removes all missing values by dropping them from the dataframe
def handle_missing_values(df):
    return df.dropna(subset=["review_text"])

#removes all duplicate values by dropping them from the dataframe
def remove_duplicates(df):
    return df.drop_duplicates(subset=["recommendationid"])


#Logic for this function was self-devised and given to ChatGPT for the code syntax for better efficiency.
def feature_engineering(df):
    df["cleaned_review"] = df["review_text"].apply(clean_text)
    df["review_length"] = df["cleaned_review"].apply(lambda x: len(x.split()))
    df["sentiment_label"] = df["voted_up"].apply(lambda x: "positive" if x else "negative")
    df["review_date"] = pd.to_datetime(df["timestamp_created"], unit="s", errors="coerce")
    return df

#performing preprocessing to gain a clean dataframe.
def preprocess_data(reviews):
    df = pd.DataFrame(reviews)

    df = handle_missing_values(df)
    df = remove_duplicates(df)
    df = feature_engineering(df)

    return df

## Step 8: Saving Processed Data to CSV

This function exports the processed dataset to a CSV file.

Saving the output as a CSV file is useful for:
- inspection of the final processed dataset
- backup of the transformed data
- further use in other tools or software

This step also helps verify that the preprocessing stage has completed successfully.

In [6]:
def save_to_csv(df, filename="steam_reviews.csv"):
    df.to_csv(filename, index=False, encoding="utf-8-sig")

## Step 9: Loading Processed Data into SQLite

This function loads the processed review data into a SQLite database table.

Database loading is an important part of the assignment requirement because the task asks for the transformed data to be loaded into an appropriate database. SQLite has been chosen because it is lightweight, simple to use, and suitable for a local academic project.

The function creates or replaces a table in the database and stores the processed data for later querying and validation.

In [7]:
def load_to_sqlite(df, database_name="steam_reviews.db", table_name="reviews"):
    connection = sqlite3.connect(database_name)
    df.to_sql(table_name, connection, if_exists="replace", index=False)
    connection.close()
    print(f"Loaded data into SQLite database: {database_name}, table: {table_name}")


## Step 10: Running the Full Pipeline

This section executes the full pipeline using a selected Steam application ID.

The process carried out here is:
1. retrieve reviews from Steam
2. preprocess and transform the data
3. display a preview of the cleaned dataset
4. save the processed output to CSV
5. load the processed output into SQLite

This provides a complete demonstration of the end-to-end workflow from acquisition to storage.

In [9]:
#Main function is AI generated since no logic was required here but just function calling.

if __name__ == "__main__":
    # Example app IDs: 570 = Dota 2, 730 = Counter-Strike 2, 440 = Team Fortress 2

    app_id = 570
    raw_reviews = get_steam_reviews(app_id=app_id, max_reviews=200, language="english", review_type="all")

    print(f"Total raw reviews acquired: {len(raw_reviews)}")

    df = preprocess_data(raw_reviews)

    print(f"Total cleaned reviews: {len(df)}")
    print(df.head())

    save_to_csv(df, filename="steam_reviews.csv")
    load_to_sqlite(df, database_name="steam_reviews.db", table_name="steam_reviews")

Total raw reviews acquired: 200
Total cleaned reviews: 200
  recommendationid                                        review_text  \
0        222789413                                                yES   
1        222787681  Yes, made me the worst person i can be. Best g...   
2        222787048                                               good   
3        222786964                                          good game   
4        222786688  This game is terrible and so is the toxic comm...   

   voted_up  votes_up  votes_funny  comment_count  steam_purchase  \
0      True         0            0              0            True   
1      True         0            0              0            True   
2      True         0            0              0            True   
3      True         0            0              0            True   
4     False         0            0              0            True   

   received_for_free  written_during_early_access language  ...  \
0              False